Web scraping is the automated process of extracting data from websites. It's like having a robot browse the internet for you and collect specific information.

Importance of Web Scraping:
- Data Collection: It's crucial for gathering large datasets for analysis, research, and machine learning. Businesses use it for market research, competitor analysis, and lead generation.
- Monitoring: Companies can monitor product prices, news, social media trends, or competitor activities in real-time.
Content Aggregation: News aggregators, job boards, and comparison shopping sites rely on web scraping to collect and display information from various sources.
- Academic Research: Researchers use it to collect data for studies in various fields, from social sciences to economics.

Precautions and Limitations:
- Legality and Ethics: Always check a website's robots.txt file (e.g., www.example.com/robots.txt) and terms of service. Many websites prohibit scraping, and doing so without permission can lead to legal issues or your IP being blocked.
- Website Changes: Websites frequently change their structure (HTML). This can break your scraping scripts, requiring constant maintenance.
- Bot Detection: Websites use various techniques to detect and block automated scraping, such as CAPTCHAs, IP blocking, and sophisticated bot detection software. This can lead to 403 Forbidden errors, as seen in a previous cell.
- Server Load: Aggressive scraping can put a heavy load on a website's server, potentially slowing it down or even crashing it. Be considerate and implement delays between requests.
- Data Quality: Scraped data might be messy, incomplete, or contain errors, requiring significant cleaning and pre-processing.
- Dynamic Content: Many modern websites use JavaScript to load content dynamically, which can be challenging for basic scrapers. Libraries like Selenium (which we used in a previous cell) are often needed to handle such cases, as they simulate a full browser environment.
- Copyright: The data you scrape might be copyrighted. Be mindful of how you use and disseminate the information you collect.

In summary, web scraping is a powerful tool, but it must be used responsibly, ethically, and with an understanding of its technical challenges and legal implications.

Task:

We want to use some web-scraping from a website. Specifically, we want to extract PDF reports and extract their data.

We will use as example the following:

https://www.gao.gov/products/gao-24-106413

Prompt:

Using Python extract all the PDF links from the website: https://www.gao.gov/products/gao-24-106413. Create a Pandas dataframe with the following column pdf_link which for every row has each extracted link.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

url = 'https://www.gao.gov/products/gao-24-106413'

try:
    response = requests.get(url)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    soup = BeautifulSoup(response.content, 'html.parser')

    pdf_links = []
    for link in soup.find_all('a', href=True):
        href = link['href']
        if href.endswith('.pdf'):
            # Resolve relative URLs to absolute URLs
            absolute_pdf_link = urljoin(url, href)
            pdf_links.append(absolute_pdf_link)

    if pdf_links:
        df = pd.DataFrame({'pdf_link': pdf_links})
        print("Successfully extracted PDF links and created DataFrame:")
        print(df)
    else:
        print("No PDF links found on the page.")
        df = pd.DataFrame(columns=['pdf_link']) # Create an empty DataFrame if no links are found

except requests.exceptions.RequestException as e:
    print(f"Error fetching the URL: {e}")
    df = pd.DataFrame(columns=['pdf_link'])
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    df = pd.DataFrame(columns=['pdf_link'])

Error fetching the URL: 403 Client Error: Forbidden for url: https://www.gao.gov/products/gao-24-106413


You may encounter connection errors in some cases (e.g., the website blocking the connection due to bot detection). An alternative may be to save the file as a mhtml and then run the prompt directing to the file instead of the webpage link.

Revised prompt:

1. How do I upload a file to the Colab files sidebar?

2. Using Python extract all the PDF links from the website mhtml file: "GAO-Safey and Health.html". Create a Pandas dataframe with the following column pdf_link which for every row has each extracted link.

There may be some iteration:
1. Each link should be its own row.

2. There should be more than PDF link found.

After inspecting spot checking some of the links and the dataframe is correct let's go to the next step. We want to extract the data from the PDF into a new column.

Prompt:

In the dataframe, create a new column called raw_text. Iterate through each row and for each link extract the pdf text and place it in the corresponding raw_text column.

The previous prompt may be unsuccessful and Gen AI may try to diagnose the issue in various ways (e.g., diagnosing the connection is valid). In some cases it may recognize that this is a dynamic website (e.g., using Javascript to submit requests) or may that the downlaod is being detected as a bot and correct the code. We may need to use the prompt below if it goes into a diagnose loop.

Prompt: We need to use the Selenium library to mimic how a browser behaves as this may be a dynamic website or the site blocking our web-scraping due to potential of being detected as a bot.

### Using Selenium to mimic browser behavior

Since direct `requests` were blocked, we will use Selenium to simulate a web browser. This allows us to interact with dynamic content and potentially bypass bot detection mechanisms by making the requests appear more like those from a human user.

First, we need to install Selenium and `webdriver_manager` to automatically handle the Chrome driver setup.

In [ ]:
# Install necessary libraries
!pip install selenium
!pip install webdriver-manager

# Install Chrome browser using .deb package
!apt-get update
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get install -f # Install dependencies

# Import necessary modules
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import os
from urllib.parse import urljoin

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
--2026-08-20 22:48:36--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Resolving dl.google.com (dl.google.com)... 74.125.137.190, 74.125.137.136, 74.125.137.93, ..

Next, we'll set up a headless Chrome browser using `webdriver_manager` and load the MHTML file. This allows us to parse the page content as if it were rendered in a browser.

In [ ]:
# Set up Selenium WebDriver for Chrome

# Configure Chrome options for headless execution
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')       # Run Chrome in headless mode (no GUI)
chrome_options.add_argument('--no-sandbox')     # Required for running in some environments
chrome_options.add_argument('--disable-dev-shm-usage') # Overcome limited resource problems

# Initialize the Chrome WebDriver using webdriver_manager
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

# The target URL to scrape
url = 'https://www.gao.gov/products/gao-24-106413'

try:
    # Open the website in the browser
    driver.get(url)

    # Get the page source after rendering (includes dynamically loaded content)
    page_source = driver.page_source

    # Use BeautifulSoup to parse the page source
    soup = BeautifulSoup(page_source, 'html.parser')

    pdf_links = []
    for link in soup.find_all('a', href=True):
        href = link['href']
        if href.endswith('.pdf'):
            # Resolve relative URLs to absolute URLs using the original website's base URL
            absolute_pdf_link = urljoin(url, href)
            pdf_links.append(absolute_pdf_link)

    # Create a Pandas DataFrame
    if pdf_links:
        df = pd.DataFrame({'pdf_link': pdf_links})
        print("Successfully extracted PDF links and created DataFrame:")
        display(df)
    else:
        print("No PDF links found on the page.")
        df = pd.DataFrame(columns=['pdf_link'])

except Exception as e:
    print(f"An error occurred while scraping: {e}")
    df = pd.DataFrame(columns=['pdf_link'])
finally:
    # Close the browser
    driver.quit()


Successfully extracted PDF links and created DataFrame:


,pdf_link
0,https://www.gao.gov/assets/gao-24-106413-highl...
1,https://www.gao.gov/assets/gao-24-106413.pdf
2,https://www.gao.gov/assets/880/872363.pdf


In [ ]:
import os
import time
import requests # requests is not strictly needed for the new approach but might be kept for general utility
from PyPDF2 import PdfReader
import io
from tqdm.notebook import tqdm

# Ensure Selenium and webdriver-manager are installed for this cell
# This makes the cell self-contained, in case of kernel restarts or independent execution.
!pip install selenium webdriver-manager PyPDF2

# Import Selenium related modules after installation
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


# Add a tqdm progress bar to pandas apply
tqdm.pandas()

def extract_text_from_pdf_with_selenium(pdf_url):
    """
    Downloads a PDF from a URL using Selenium and extracts its text content.
    Handles 403 Forbidden errors by mimicking browser download.
    """
    # Create a unique temporary directory for each download to avoid conflicts
    download_dir = os.path.join(os.getcwd(), f'pdf_downloads_{os.getpid()}_{time.time()}')
    os.makedirs(download_dir, exist_ok=True)

    driver = None # Initialize driver to None for proper cleanup in finally block

    try:
        # Configure Chrome options for headless execution AND PDF downloads
        chrome_options = webdriver.ChromeOptions()
        chrome_options.add_argument('--headless')
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--disable-dev-shm-usage')

        # Set preferences for PDF download
        prefs = {
            "download.default_directory": download_dir,
            "download.prompt_for_download": False, # To avoid download dialog
            "download.directory_upgrade": True,
            "plugins.always_open_pdf_externally": True # Important for auto-downloading PDFs
        }
        chrome_options.add_experimental_option("prefs", prefs)

        # Initialize the Chrome WebDriver for this specific download
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

        # Navigate to the PDF URL, which should trigger a download
        driver.get(pdf_url)

        downloaded_file = None
        wait_attempts = 0
        max_wait_attempts = 60 # Wait up to 60 seconds for the download to complete
        timeout_seconds = 1 # Check for file every second

        while wait_attempts < max_wait_attempts:
            files_in_dir = os.listdir(download_dir)
            if files_in_dir:
                # Filter out partial download files like .crdownload or .tmp
                complete_files = [f for f in files_in_dir if not f.endswith(('.crdownload', '.tmp'))]
                if complete_files:
                    # Assuming the first complete file found is the one we want
                    downloaded_file = os.path.join(download_dir, complete_files[0])
                    break # File downloaded and appears complete
            time.sleep(timeout_seconds)
            wait_attempts += 1

        if downloaded_file and os.path.exists(downloaded_file):
            text = ""
            with open(downloaded_file, 'rb') as f:
                reader = PdfReader(f)
                for page_num in range(len(reader.pages)):
                    page = reader.pages[page_num]
                    text += page.extract_text() or ""
            return text.strip()
        else:
            print(f"Failed to download or find complete PDF for {pdf_url} after {max_wait_attempts} seconds.")
            return None

    except Exception as e:
        print(f"An error occurred while processing PDF {pdf_url}: {e}")
        return None
    finally:
        if driver:
            driver.quit() # Ensure the driver is closed after use
        # Clean up the temporary download directory and its contents
        if os.path.exists(download_dir):
            for f in os.listdir(download_dir):
                os.remove(os.path.join(download_dir, f))
            os.rmdir(download_dir)

# Create the 'raw_text' column by applying the new extraction function
print("Extracting text from PDF links (this might take a while due to individual Selenium runs for each PDF)...")
df['raw_text'] = df['pdf_link'].progress_apply(extract_text_from_pdf_with_selenium)

print("\nDataFrame with 'raw_text' column:")
print(df.head())

Extracting text from PDF links (this might take a while due to individual Selenium runs for each PDF)...


  0%|          | 0/3 [00:00<?, ?it/s]

An error occurred while processing PDF https://www.gao.gov/assets/gao-24-106413-highlights.pdf: [Errno 22] Invalid argument
An error occurred while processing PDF https://www.gao.gov/assets/gao-24-106413.pdf: [Errno 22] Invalid argument
An error occurred while processing PDF https://www.gao.gov/assets/880/872363.pdf: EOF marker not found

DataFrame with 'raw_text' column:
                                            pdf_link raw_text
0  https://www.gao.gov/assets/gao-24-106413-highl...     None
1       https://www.gao.gov/assets/gao-24-106413.pdf     None
2          https://www.gao.gov/assets/880/872363.pdf     None
